In [7]:
import torch
from latent_models.latent_utils import get_latent_model

class Args: pass
args = Args()
args.enc_dec_model          = "facebook/bart-base"
args.num_encoder_latents    = 1
args.num_decoder_latents    = 1
args.dim_ae                 = 128
args.num_layers             = 2
args.l2_normalize_latents   = False
args.lm_mode                = "freeze"


lm, tokenizer, config = get_latent_model(args)  

ckpt = torch.load("./model.pt", map_location="cpu")
lm.load_state_dict(ckpt["model"], strict=False)

lm.eval()

inputs = tokenizer("It didn't rained today", return_tensors="pt", truncation=True, padding=True)
enc_opts = lm.get_encoder()(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
latents  = lm.get_diffusion_latent(enc_opts, inputs.attention_mask)
print("Raw latent tensor:\n", latents)
print(latents.shape)   


Some weights of BARTForConditionalGenerationLatent were not initialized from the model checkpoint at facebook/bart-base and are newly initialized: ['perceiver_ae.perceiver_decoder.layers.1.1.4.bias', 'perceiver_ae.perceiver_decoder.final_norm.weight', 'perceiver_ae.perceiver_decoder.layers.1.1.0.bias', 'perceiver_ae.perceiver_encoder.layers.1.1.0.weight', 'perceiver_ae.perceiver_decoder.layers.0.0.to_out.weight', 'perceiver_ae.perceiver_encoder.layers.1.0.to_out.0.weight', 'perceiver_ae.perceiver_encoder.layers.0.1.0.weight', 'perceiver_ae.perceiver_encoder.final_norm.weight', 'perceiver_ae.perceiver_encoder.layers.1.0.norm_latents.weight', 'perceiver_ae.perceiver_decoder.layers.1.0.to_out.weight', 'perceiver_ae.perceiver_decoder.layers.1.0.query_norm.gamma', 'perceiver_ae.perceiver_encoder.layers.1.1.0.bias', 'perceiver_ae.perceiver_encoder.layers.0.0.key_norm.gamma', 'perceiver_ae.perceiver_decoder.layers.1.0.to_out.bias', 'perceiver_ae.perceiver_encoder.layers.0.0.to_out.0.weight', 

Trainable: perceiver_ae.perceiver_encoder.latents
Trainable: perceiver_ae.perceiver_encoder.pos_emb.emb.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm.bias
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm_latents.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.norm_latents.bias
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.query_norm.gamma
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.key_norm.gamma
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_q.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.latent_to_kv.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_kv.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_out.0.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.0.to_out.0.bias
Trainable: perceiver_ae.perceiver_encoder.layers.0.1.0.weight
Trainable: perceiver_ae.perceiver_encoder.layers.0.1.0.bias
Trainable: percei

In [22]:
import re

with open("./output.txt") as f:
    text = f.read()

def extract_blocks(text):
    blocks = []
    idx = 0
    while True:
        pos = text.find("responses", idx)
        if pos == -1:
            break
        start = text.find("[", pos)
        if start == -1:
            break
        depth = 0
        for i, ch in enumerate(text[start:], start):
            if ch == "[":
                depth += 1
            elif ch == "]":
                depth -= 1
                if depth == 0:
                    blocks.append(text[start:i+1])
                    idx = i + 1
                    break
        else:
            break
    return blocks

blocks = extract_blocks(text)
responses_lists = []
for blk in blocks:
    try:
        responses = eval(blk, {"tensor": lambda *args, **kwargs: None})
        responses_lists.append(responses)
    except:
        pass

from pprint import pprint
pprint(responses_lists)


[[('radio wave communications',
   [-3.650303602218628, -4.025141716003418, -1.7355194091796875],
   None,
   0.0),
  ('wireless data transfer',
   [-1.789168119430542, -1.6794599294662476, -0.6588299870491028],
   None,
   0.0),
  ('portable communications',
   [-3.908777952194214, -2.921856164932251],
   None,
   0.0),
  ('communication', [-3.1746928691864014], None, 0.0),
  ('communication', [-3.1746928691864014], None, 0.0),
  ('near field communication',
   [-4.673958778381348, -1.017874002456665, -0.503303050994873],
   None,
   0.0),
  ('phones', [-4.317170143127441], None, 0.0),
  ('mobile phones', [-1.1736953258514404, -0.434695839881897], None, 0.0),
  ('communication between devices',
   [-3.1746928691864014, -1.8701362609863281, -1.5504412651062012],
   None,
   0.0),
  ('bluetooth', [-4.2521562576293945], None, 0.0)],
 [('the address', [-0.7398093938827515, -3.4815847873687744], None, 0.0),
  ('a letter head',
   [-2.325260639190674, -1.8908360004425049, -2.750032901763916

In [23]:
texts_only = [[resp[0] for resp in resp_list] for resp_list in responses_lists]

from pprint import pprint
pprint(texts_only)


[['radio wave communications',
  'wireless data transfer',
  'portable communications',
  'communication',
  'communication',
  'near field communication',
  'phones',
  'mobile phones',
  'communication between devices',
  'bluetooth'],
 ['the address',
  'a letter head',
  'space',
  'the top left hand corner',
  'salutation',
  'the letter head',
  'its height',
  'the capital',
  'the tail',
  'the space between the capital letter and the lower case letter'],
 ['count of russia and les mizhi',
  'lord of the rings',
  "Ivan Turgenev's Fathers and Sons",
  'war and peace',
  'war and peace',
  'War and Peace',
  'war and peace',
  'the 3rd book of the "Gogol" trilogy "The Inspector General"',
  'war and peace',
  'war and peace'],
 ['Joseph Goebbels',
  'joachim stiehl',
  'Josef Goebbels',
  'joseph goebbels',
  'Goebbels',
  'josef goebbels',
  'martin bormann',
  'Adolf Hitler',
  'goebbels',
  'joachim stahlecker'],
 ['new york',
  'dublin',
  'paris',
  'NEW YORK',
  'london',


In [24]:
# save the tensors to a file
with open("tensors.txt", "w") as f:
    for group in texts_only:
        for text in group:
            print(text)
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
            enc_opts = lm.get_encoder()(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask)
            latents  = lm.get_diffusion_latent(enc_opts, inputs.attention_mask)
            print("Raw latent tensor:\n", latents)
            print(latents.shape)
            f.write(str(text) + "\n")
            f.write(str(latents.tolist()) + "\n")
            f.write("\n")



radio wave communications
Raw latent tensor:
 tensor([[[ 3.9490e-01, -1.2839e-01,  4.7641e-01,  2.0292e+00,  1.2063e+00,
          -2.7486e-01, -1.9379e+00, -1.1461e+00,  4.9106e-01,  1.3491e+00,
           5.0198e-01, -5.1076e-02,  6.9611e-01, -4.7072e-01,  6.4365e-01,
          -1.1129e+00, -2.5545e-02,  1.6407e+00,  1.9390e-01, -1.3183e+00,
           1.1245e+00, -5.5886e-01, -1.0273e+00,  4.5955e-01, -1.0857e-01,
          -1.9933e-03, -2.4881e-01, -1.5676e+00,  1.3450e+00, -4.8132e-01,
          -8.8016e-02, -2.7592e-01, -4.8109e-01,  9.8414e-01,  1.9768e+00,
          -4.5888e-01, -1.2179e+00, -5.1609e-01, -1.2118e-01, -1.6647e+00,
          -6.7661e-01, -2.4060e+00, -1.3759e+00, -1.1623e+00, -2.6946e-01,
           4.0295e-01,  7.2050e-01, -1.7734e+00,  1.5029e+00,  8.1007e-01,
          -1.4071e+00, -1.5038e+00, -4.5577e-01, -5.2588e-01, -3.7935e-01,
          -3.9968e-01,  4.3714e-01,  1.9140e-01, -6.3177e-01,  7.0197e-01,
           1.5411e+00,  3.3675e-01,  1.8080e+00, -3.14